# Zomato Bangalore Restaurant Analytics
## Notebook 1: Data Loading & Cleaning

Load the Zomato CSV, clean it up, and store the processed data in a normalized SQLite database.

---

### Dataset Overview
| Property | Detail |
|---|---|
| Source | Zomato Bangalore Restaurants Dataset |
| Rows | ~71,730 |
| Columns | 17 |
| Size | ~574 MB |

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import sqlite3
import re
import ast
import warnings
warnings.filterwarnings('ignore')

print('All imports successful')

---
## 1. Load Raw Data

In [ ]:
# Load the CSV
df_raw = pd.read_csv('../zomato.csv', encoding='latin-1')

print(f'Dataset Shape: {df_raw.shape}')
print(f'Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]}')
df_raw.head(3)

In [ ]:
# Column info
print('\nColumn Data Types & Non-Null Counts:')
print('=' * 55)
df_raw.info()

In [ ]:
# Missing values summary
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('\nColumns with Missing Values:')
missing_df

---
## 2. Data Cleaning

In [ ]:
# Work on a copy
df = df_raw.copy()

# -------------------------------------------------------
# 2a. Clean the 'rate' column
#     Values like '3.8/5', 'NEW', '-', 'nan'
# -------------------------------------------------------
def clean_rating(val):
    """Extract numeric rating from strings like '3.8/5'."""
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if val in ['NEW', '-', '', 'nan']:
        return np.nan
    match = re.match(r'([\d.]+)', val)
    if match:
        rating = float(match.group(1))
        return rating if 1.0 <= rating <= 5.0 else np.nan
    return np.nan

df['rating'] = df['rate'].apply(clean_rating)

print(f'Ratings cleaned: {df["rating"].notna().sum():,} valid ratings out of {len(df):,} rows')
print(f'   Rating range: {df["rating"].min()} – {df["rating"].max()}')
print(f'   Mean rating: {df["rating"].mean():.2f}')

In [ ]:
# -------------------------------------------------------
# 2b. Clean 'approx_cost(for two people)'
#     Remove commas, convert to int
# -------------------------------------------------------
def clean_cost(val):
    """Remove commas and convert to integer."""
    if pd.isna(val):
        return np.nan
    val = str(val).replace(',', '').strip()
    try:
        cost = int(float(val))
        return cost if cost > 0 else np.nan
    except ValueError:
        return np.nan

df['approx_cost'] = df['approx_cost(for two people)'].apply(clean_cost)

print(f'Cost cleaned: {df["approx_cost"].notna().sum():,} valid entries')
print(f'   Cost range: ₹{df["approx_cost"].min():.0f} – ₹{df["approx_cost"].max():.0f}')

In [ ]:
# -------------------------------------------------------
# 2c. Clean boolean columns
# -------------------------------------------------------
df['online_order'] = df['online_order'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)
df['book_table']   = df['book_table'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

print(f'Online order: {df["online_order"].sum():,} restaurants offer it ({df["online_order"].mean()*100:.1f}%)')
print(f'Table booking: {df["book_table"].sum():,} restaurants offer it ({df["book_table"].mean()*100:.1f}%)')

In [ ]:
# -------------------------------------------------------
# 2d. Clean votes
# -------------------------------------------------------
df['votes'] = pd.to_numeric(df['votes'], errors='coerce').fillna(0).astype(int)

print(f'Votes range: {df["votes"].min()} – {df["votes"].max():,}')
print(f'   Mean votes: {df["votes"].mean():.0f}')

In [ ]:
# -------------------------------------------------------
# 2e. Trim text columns
# -------------------------------------------------------
text_cols = ['name', 'address', 'phone', 'location', 'rest_type', 'dish_liked', 'cuisines']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace('nan', np.nan)

print('Text columns trimmed')

In [ ]:
# -------------------------------------------------------
# 2f. Rename columns for clarity
# -------------------------------------------------------
df = df.rename(columns={
    'listed_in(type)': 'listed_type',
    'listed_in(city)': 'city'
})

print('Columns renamed')
print(f'\nFinal columns: {list(df.columns)}')

In [ ]:
# -------------------------------------------------------
# 2g. Remove exact duplicates
# -------------------------------------------------------
before = len(df)
df = df.drop_duplicates(subset=['name', 'address'], keep='first')
after = len(df)

print(f'Duplicates removed: {before - after:,} rows')
print(f'   Remaining rows: {after:,}')

---
## 3. Create SQLite Database with Normalized Tables

In [ ]:
# ============================================================
# Create the SQLite database
# ============================================================
DB_PATH = '../zomato_bangalore.db'

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Read and execute the schema SQL
with open('../sql_queries/01_create_tables.sql', 'r') as f:
    schema_sql = f.read()

cursor.executescript(schema_sql)
conn.commit()

print('Database created and schema applied')

In [ ]:
# ============================================================
# 3a. Insert data into 'restaurants' table
# ============================================================
restaurants_df = df[[
    'name', 'url', 'address', 'phone', 'location', 'city',
    'rest_type', 'online_order', 'book_table', 'rating',
    'votes', 'approx_cost', 'listed_type'
]].copy()

restaurants_df.to_sql('restaurants', conn, if_exists='replace', index=True, index_label='id')

count = cursor.execute('SELECT COUNT(*) FROM restaurants').fetchone()[0]
print(f'Inserted {count:,} rows into restaurants table')

In [ ]:
# ============================================================
# 3b. Normalize cuisines → restaurant_cuisines table
# ============================================================
cuisine_rows = []
for idx, row in df.iterrows():
    if pd.notna(row.get('cuisines')):
        cuisines = [c.strip() for c in str(row['cuisines']).split(',') if c.strip()]
        for cuisine in cuisines:
            cuisine_rows.append({'restaurant_id': idx, 'cuisine': cuisine})

cuisine_df = pd.DataFrame(cuisine_rows)
cuisine_df.to_sql('restaurant_cuisines', conn, if_exists='replace', index=True, index_label='id')

print(f'Inserted {len(cuisine_df):,} rows into restaurant_cuisines table')
print(f'   Unique cuisines: {cuisine_df["cuisine"].nunique()}')

In [ ]:
# ============================================================
# 3c. Normalize dishes → restaurant_dishes table
# ============================================================
dish_rows = []
for idx, row in df.iterrows():
    if pd.notna(row.get('dish_liked')):
        dishes = [d.strip() for d in str(row['dish_liked']).split(',') if d.strip()]
        for dish in dishes:
            dish_rows.append({'restaurant_id': idx, 'dish': dish})

dish_df = pd.DataFrame(dish_rows)
dish_df.to_sql('restaurant_dishes', conn, if_exists='replace', index=True, index_label='id')

print(f'Inserted {len(dish_df):,} rows into restaurant_dishes table')
print(f'   Unique dishes: {dish_df["dish"].nunique()}')

In [ ]:
# ============================================================
# 3d. Parse reviews → reviews table
#     reviews_list is a string repr of list of tuples:
#     [('Rated 3.0', 'review text'), ...]
# ============================================================
review_rows = []
parse_errors = 0

for idx, row in df.iterrows():
    reviews_str = row.get('reviews_list')
    if pd.isna(reviews_str) or str(reviews_str).strip() in ['', '[]', 'nan']:
        continue
    try:
        reviews = ast.literal_eval(str(reviews_str))
        for rated_str, review_text in reviews:
            # Extract numeric rating from 'Rated 3.0'
            match = re.search(r'([\d.]+)', str(rated_str))
            rev_rating = float(match.group(1)) if match else None
            # Clean review text
            clean_text = re.sub(r'^RATED\s*', '', str(review_text).strip())
            review_rows.append({
                'restaurant_id': idx,
                'reviewer_rating': rev_rating,
                'review_text': clean_text[:2000]  # cap length
            })
    except Exception:
        parse_errors += 1

review_df = pd.DataFrame(review_rows)
review_df.to_sql('reviews', conn, if_exists='replace', index=True, index_label='id')

print(f'Inserted {len(review_df):,} reviews into reviews table')
print(f'   Parse errors (skipped): {parse_errors}')

In [ ]:
# ============================================================
# 3e. Run cleaning SQL queries on the database
# ============================================================
with open('../sql_queries/02_data_cleaning.sql', 'r') as f:
    cleaning_sql = f.read()

# Execute each statement separately
for statement in cleaning_sql.split(';'):
    stmt = statement.strip()
    if stmt and not stmt.startswith('--'):
        try:
            cursor.execute(stmt)
        except Exception as e:
            pass  # SELECT statements at the end

conn.commit()
print('Cleaning queries executed')

In [ ]:
# ============================================================
# 3f. Final data quality report
# ============================================================
print('FINAL DATABASE SUMMARY')
print('=' * 50)

tables = ['restaurants', 'restaurant_cuisines', 'restaurant_dishes', 'reviews']
for table in tables:
    count = cursor.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'   {table:25s} → {count:>10,} rows')

# NULL check for restaurants
null_check = pd.read_sql_query('''
    SELECT
        SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS null_ratings,
        SUM(CASE WHEN approx_cost IS NULL THEN 1 ELSE 0 END) AS null_cost,
        SUM(CASE WHEN location IS NULL OR location = '' THEN 1 ELSE 0 END) AS null_location,
        COUNT(*) AS total
    FROM restaurants
''', conn)

print(f'\nNULL Summary:')
print(null_check.to_string(index=False))

conn.close()
print(f'\nDatabase saved to: {DB_PATH}')
print('\nDone. The database is ready — move on to Notebook 2 for the SQL queries.')